# Day 8 · Exercise 2: Single-Label Classifier

**What you'll build:** `classify(text: str, labels: list[str], model: str) -> str` — a zero-shot classifier that sends a constrained-output prompt to an Ollama model and returns exactly one validated label from the list you provide.

**Why it matters:** Reliable classification is the foundation of every text-routing, tagging, and triage system; mastering the constrained-output prompt pattern means you can classify any text without training a single model.

## Your Implementation

In [ ]:
import ollama

MODEL = "llama3.2"

# ── Define CLASSIFY_SYSTEM_PROMPT here (module-level constant) ──────────────
# Use a triple-quoted string with a {labels} placeholder.
# Tell the model to reply with exactly one label and nothing else.
CLASSIFY_SYSTEM_PROMPT = ""  # YOUR CODE HERE


def classify(text: str, labels: list[str], model: str = MODEL) -> str:
    """Classify text into exactly one of the given labels using zero-shot prompting.

    Args:
        text:   The text to classify.
        labels: Valid output labels (e.g. ["billing", "technical", "general"]).
        model:  Ollama model name (default: module-level MODEL constant).

    Returns:
        One label from the labels list, lowercased and stripped.

    Raises:
        ValueError: If the model returns a value that is not in the labels list.

    Example:
        >>> classify("My invoice is wrong.", ["billing", "technical", "general"])
        'billing'
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
import unittest.mock as _mock

_PASS, _FAIL = '✅', '❌'

async def _run_checks():
    score, total = 0, 4

    # Check 1: function exists and is callable
    try:
        assert callable(classify), 'classify is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: classify is defined and callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: returns a string that is in the label list
    try:
        _labels = ['billing', 'technical', 'general']
        _result = classify('My invoice shows the wrong amount.', _labels, MODEL)
        assert isinstance(_result, str), f'expected str, got {type(_result).__name__}'
        assert _result in _labels, f'returned {_result!r} which is not in {_labels}'
        print(f'{_PASS} Check 2/{total}: classify() returned a valid label ({_result!r})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')

    # Check 3: output is lowercased (normalised)
    try:
        _labels = ['positive', 'negative', 'neutral']
        _result = classify('I love this product!', _labels, MODEL)
        assert _result == _result.lower(), f'label {_result!r} is not all lowercase'
        assert _result in _labels, f'returned {_result!r} which is not in {_labels}'
        print(f'{_PASS} Check 3/{total}: output is lowercase and in label list ({_result!r})')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: raises ValueError when model output is not a valid label
    try:
        _fake = {'message': {'content': 'DEFINITELY_NOT_A_LABEL'}}
        with _mock.patch('ollama.chat', return_value=_fake):
            _raised = False
            try:
                classify('Some text.', ['billing', 'technical', 'general'])
            except ValueError:
                _raised = True
        assert _raised, 'classify() should raise ValueError for an out-of-list model response'
        print(f'{_PASS} Check 4/{total}: raises ValueError for invalid model output')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
    print(f'  {score}/{total} passed.' + ('' if score == total else ' Keep going!'))

await _run_checks()

## Bonus Challenge

Right now `classify()` calls the model once and raises `ValueError` if the label is invalid. A more robust version could **retry once** before raising — useful when the model occasionally returns a stray character or punctuation mark.

Try wrapping the model call in a loop that retries up to `max_retries=2` times before raising. This foreshadows the **retry-with-backoff** pattern you will build on Day 31 (Resilience & Error Handling) when connecting to live APIs that impose rate limits.

```python
def classify(text, labels, model=MODEL, max_retries=2):
    for attempt in range(1, max_retries + 1):
        # ... call the model and validate ...
        # if valid: return immediately
        # if not valid: loop again (up to max_retries)
    raise ValueError(f"Model failed to return a valid label after {max_retries} attempts")
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
import ollama

MODEL = "llama3.2"

CLASSIFY_SYSTEM_PROMPT = """You are a text classifier.

Classify the text into exactly one of these labels:
{labels}

Rules:
- Reply with only the label, exactly as written above.
- No punctuation. No explanation. No extra words.
- If you are unsure, pick the closest match."""


def classify(text: str, labels: list[str], model: str = MODEL) -> str:
    """Classify text into exactly one of the given labels using zero-shot prompting.

    Args:
        text:   The text to classify.
        labels: Valid output labels (e.g. ["billing", "technical", "general"]).
        model:  Ollama model name (default: module-level MODEL constant).

    Returns:
        One label from the labels list, lowercased and stripped.

    Raises:
        ValueError: If the model returns a value that is not in the labels list.

    Example:
        >>> classify("My invoice is wrong.", ["billing", "technical", "general"])
        'billing'
    """
    label_str = ", ".join(labels)
    system_msg = CLASSIFY_SYSTEM_PROMPT.format(labels=label_str)

    messages = [
        {"role": "system", "content": system_msg},
        {
            "role": "user",
            "content": (
                f"Text to classify:\n{text}\n\n"
                f"Reply with exactly one label from: {label_str}"
            ),
        },
    ]

    response = ollama.chat(model=model, messages=messages)
    raw = response["message"]["content"]

    result = raw.strip().lower()
    valid = [lb.lower() for lb in labels]
    if result not in valid:
        raise ValueError(
            f"Model returned {raw!r} — not in label list {labels}"
        )
    return result
```

**Why this works:** The `CLASSIFY_SYSTEM_PROMPT` constant uses a `{labels}` placeholder so the same template covers any classification task — the real label list is injected at call time via `.format()`. The constrained-output rules (enumerate exact labels, forbid explanation, repeat the constraint in the user turn) prevent the model from returning prose instead of a single word. Finally, `.strip().lower()` normalises the raw response before checking membership, so minor inconsistencies like trailing spaces or mixed capitalisation never cause a false `ValueError`.
</details>